# ЛР-1. Примитивные операции над векторами (CUDA)
**Вариант 1 — сложение векторов**

Требования:
- ввод из stdin, вывод в stdout;
- тип `double`, точность 10⁻¹⁰, формат вывода `%.10e` (числа через пробел в одну строку);
- ограничение `n < 2²⁵`;
- количество вычислительных потоков фиксировано и не зависит от `n` (сетка 256×256 потоков + grid-stride loop);
- программа не падает ни при каких условиях: при ошибке печатает `ERROR: ...` в stderr и завершается с кодом возврата 0.

**Перед запуском: Runtime → Change runtime type → Hardware accelerator: GPU (T4).**


In [ ]:
!nvidia-smi
import os
os.environ['PATH'] = '/usr/local/cuda/bin:' + os.environ['PATH']
!nvcc --version


## Исходный код `lab1.cu`

In [ ]:
%%writefile lab1.cu
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <cerrno>
#include <cctype>

#define MAX_N (1LL << 25)
#define BLOCKS 256
#define THREADS 256

__global__ void vecAddKernel(const double *a, const double *b, double *c, long long n)
{
    long long stride = (long long)gridDim.x * blockDim.x;
    for (long long i = (long long)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride)
        c[i] = a[i] + b[i];
}

static void fail(const char *msg)
{
    fprintf(stderr, "ERROR: %s\n", msg);
    exit(0);
}

static char *readAllInput(void)
{
    size_t cap = 1 << 20, size = 0;
    char *buf = (char *)malloc(cap);
    if (!buf)
        return NULL;
    for (;;) {
        if (size == cap) {
            char *nb = (char *)realloc(buf, cap << 1);
            if (!nb) {
                free(buf);
                return NULL;
            }
            buf = nb;
            cap <<= 1;
        }
        size_t r = fread(buf + size, 1, cap - size, stdin);
        if (r == 0)
            break;
        size += r;
    }
    if (ferror(stdin)) {
        free(buf);
        return NULL;
    }
    buf[size] = '\0';
    return buf;
}

static long long parseN(char **pp)
{
    char *p = *pp;
    while (*p != '\0' && isspace((unsigned char)*p))
        p++;
    if (*p == '\0')
        return -1;
    errno = 0;
    char *end = NULL;
    long long n = strtoll(p, &end, 10);
    if (end == p || errno == ERANGE)
        return -1;
    if (*end != '\0' && !isspace((unsigned char)*end))
        return -1;
    if (n < 0 || n >= MAX_N)
        return -1;
    *pp = end;
    return n;
}

static int parseVec(char **pp, double *v, long long n)
{
    char *p = *pp;
    for (long long i = 0; i < n; i++) {
        char *end = NULL;
        double x = strtod(p, &end);
        if (end == p)
            return -1;
        v[i] = x;
        p = end;
    }
    *pp = p;
    return 0;
}

int main(void)
{
    char *buf = readAllInput();
    if (!buf)
        fail("cannot read input");

    char *p = buf;
    long long n = parseN(&p);
    if (n < 0)
        fail("invalid vector size");

    if (n == 0) {
        free(buf);
        return 0;
    }

    double *a = (double *)malloc((size_t)n * sizeof(double));
    double *b = (double *)malloc((size_t)n * sizeof(double));
    double *c = (double *)malloc((size_t)n * sizeof(double));
    if (!a || !b || !c)
        fail("out of memory");

    if (parseVec(&p, a, n) != 0 || parseVec(&p, b, n) != 0)
        fail("invalid vector data");

    free(buf);

    cudaError_t err;
    int devCount = 0;
    if ((err = cudaGetDeviceCount(&devCount)) != cudaSuccess || devCount < 1)
        fail("no CUDA device available");

    double *da = NULL, *db = NULL, *dc = NULL;
    if ((err = cudaMalloc((void **)&da, (size_t)n * sizeof(double))) != cudaSuccess)
        fail("cudaMalloc failed");
    if ((err = cudaMalloc((void **)&db, (size_t)n * sizeof(double))) != cudaSuccess)
        fail("cudaMalloc failed");
    if ((err = cudaMalloc((void **)&dc, (size_t)n * sizeof(double))) != cudaSuccess)
        fail("cudaMalloc failed");

    if ((err = cudaMemcpy(da, a, (size_t)n * sizeof(double), cudaMemcpyHostToDevice)) != cudaSuccess)
        fail("cudaMemcpy host to device failed");
    if ((err = cudaMemcpy(db, b, (size_t)n * sizeof(double), cudaMemcpyHostToDevice)) != cudaSuccess)
        fail("cudaMemcpy host to device failed");

    vecAddKernel<<<BLOCKS, THREADS>>>(da, db, dc, n);
    if ((err = cudaGetLastError()) != cudaSuccess)
        fail("kernel launch failed");

    if ((err = cudaMemcpy(c, dc, (size_t)n * sizeof(double), cudaMemcpyDeviceToHost)) != cudaSuccess)
        fail("cudaMemcpy device to host failed");
    if ((err = cudaDeviceSynchronize()) != cudaSuccess)
        fail("cudaDeviceSynchronize failed");

    cudaFree(da);
    cudaFree(db);
    cudaFree(dc);

    size_t obufCap = 1 << 20;
    char *obuf = (char *)malloc(obufCap);
    if (!obuf)
        fail("out of memory");
    size_t pos = 0;
    for (long long i = 0; i < n; i++) {
        if (pos + 64 > obufCap) {
            if (fwrite(obuf, 1, pos, stdout) != pos)
                fail("write to stdout failed");
            pos = 0;
        }
        if (i > 0)
            obuf[pos++] = ' ';
        int w = snprintf(obuf + pos, 48, "%.10e", c[i]);
        if (w < 0 || w >= 48)
            fail("output formatting failed");
        pos += (size_t)w;
    }
    obuf[pos++] = '\n';
    if (fwrite(obuf, 1, pos, stdout) != pos)
        fail("write to stdout failed");

    free(obuf);
    free(a);
    free(b);
    free(c);
    return 0;
}


In [ ]:
!nvcc -Werror cross-execution-space-call -lm -o lab1 lab1.cu && echo "BUILD OK"


## Тест 1. Пример из задания

`3 / 1 2 3 / 4 5 6` → `5.0000000000e+00 7.0000000000e+00 9.0000000000e+00`

In [ ]:
import subprocess

def run(inp):
    return subprocess.run(['./lab1'], input=inp, capture_output=True, text=True)

r = run("3\n1 2 3\n4 5 6\n")
print("stdout :", repr(r.stdout))
print("stderr :", repr(r.stderr), "| returncode:", r.returncode)
expected = "5.0000000000e+00 7.0000000000e+00 9.0000000000e+00\n"
print("PASS" if r.stdout == expected and r.returncode == 0 else "FAIL")


## Тест 2. Граничные размеры: n = 1 и n = 0

In [ ]:
r = run("1\n-1.5\n2.25\n")
print("n=1 stdout:", repr(r.stdout), "rc:", r.returncode)
print("PASS" if r.stdout == "7.5000000000e-01\n" and r.returncode == 0 else "FAIL")

r = run("0\n\n\n")
print("n=0 stdout:", repr(r.stdout), "rc:", r.returncode, "stderr:", repr(r.stderr))
print("PASS" if r.returncode == 0 and r.stdout == "" and r.stderr == "" else "FAIL")


## Тест 3. Отрицательные, дробные, большие и малые значения

In [ ]:
a = [-1.5, 1e-300, 1e300, -123456.789123456]
b = [ 2.25, 2e-300, 2e300,  123456.789123456]
inp = str(len(a)) + "\n" + " ".join(map(repr, a)) + "\n" + " ".join(map(repr, b)) + "\n"
r = run(inp)
got = r.stdout.split()
exp = ["%.10e" % (x + y) for x, y in zip(a, b)]
print("rc:", r.returncode)
ok = r.returncode == 0 and len(got) == len(exp) and all(g == e for g, e in zip(got, exp))
for g, e in zip(got, exp):
    print(f"{g}  expected {e}  {'OK' if g == e else 'FAIL'}")
print("PASS" if ok else "FAIL")


## Тест 4. Максимальный размер n = 2²⁵ − 1 = 33 554 431

Значения генерируются как `k / 1000.0`, где `k` — целое: такие числа гарантированно без потерь проходят через текстовый формат `%.3f`, поэтому эталон на CPU должен совпадать с GPU **бит-в-бит** (сложение двух `double` по IEEE 754 детерминировано).

Вывод программы пишется в файл и читается `np.fromfile(sep=' ')` — чтобы не держать в памяти десятки миллионов Python-строк.

Если в Colab всё же не хватает памяти/диска — уменьшите `n` до `2**24`, но максимальное значение предпочтительнее.

In [ ]:
import numpy as np, time

n = 2**25 - 1
rng = np.random.default_rng(42)
a = rng.integers(-10**6, 10**6, n).astype(np.float64) / 1000.0
b = rng.integers(-10**6, 10**6, n).astype(np.float64) / 1000.0

t0 = time.time()
with open('big.txt', 'w') as f:
    f.write(str(n) + "\n")
    for v in (a, b):
        for i in range(0, n, 1_000_000):
            f.write(' '.join(np.char.mod('%.3f', v[i:i+1_000_000])))
            f.write('\n')
print(f"input generated in {time.time()-t0:.1f}s")

t0 = time.time()
with open('big.txt') as fin, open('out.txt', 'w') as fout:
    r = subprocess.run(['./lab1'], stdin=fin, stdout=fout,
                       stderr=subprocess.PIPE, text=True)
print(f"run time: {time.time()-t0:.1f}s | rc: {r.returncode} | stderr: {r.stderr[:200]!r}")

out = np.fromfile('out.txt', sep=' ')
ref = a + b
print("elements out/ref:", out.shape, ref.shape)
if out.shape == ref.shape:
    exact = bool(np.array_equal(out, ref))
    rel = np.abs(out - ref) / np.maximum(np.abs(ref), 1e-300)
    print("exact match:", exact)
    print("max rel err:", rel.max())
    if exact:
        print("PASS (exact)")
    elif rel.max() < 1e-10:
        print("PASS (within 1e-10 tolerance)")
    else:
        print("FAIL")
else:
    print("FAIL")
os.remove('big.txt')
os.remove('out.txt')


## Тест 5. Некорректный ввод

Ожидание: `ERROR: ...` в stderr, **код возврата 0**, отсутствие segfault.

In [ ]:
cases = {
    "пустой ввод":        "",
    "n не число":         "abc\n1 2\n",
    "n дробное":          "3.5\n1 2 3\n4 5 6\n",
    "n отрицательное":    "-3\n1 2 3\n",
    "n >= 2^25":          "33554432\n1 2\n",
    "чисел меньше n":     "3\n1 2 3\n4 5\n",
    "мусор в векторе":    "2\n1 x\n3 4\n",
}
all_ok = True
for name, inp in cases.items():
    r = run(inp)
    ok = r.returncode == 0 and r.stderr.startswith("ERROR:")
    all_ok &= ok
    print(f"{'PASS' if ok else 'FAIL'} | {name}: rc={r.returncode}, stderr={r.stderr.strip()!r}, stdout={r.stdout!r}")
print()
print("ALL PASS" if all_ok else "SOME FAILED")


## Итог

Если во всех тестах `PASS` / `BUILD OK` — работа готова к отправке:
файлы `lab1.cu` и его подпись `lab1.cu.asc` на **checkpgp@gmail.com**, тема письма: `pgp:1`.